# rd-galleries: review notebook

Exercises `python/evoca_gallery.py` through the doors the RD notebook uses. Each cell prints PASS/FAIL. The last cell opens an SDL window, so it is commented out.

In [ ]:
import sys, time; from pathlib import Path
ROOT = Path.cwd(); sys.path.insert(0, str(ROOT / 'python'))
import numpy as np
from evoca_gallery import Gallery, Entry, all_galleries
ok = lambda name, cond: print(('PASS ' if cond else 'FAIL ') + name)
print([Path(p).name for p in all_galleries('Reaction-diffusion')])

In [ ]:
# 1. open a saved gallery; the index table is its repr
PICKS = Gallery.open('rd_pareto_picks'); display(PICKS)
ok('29 picks', len(PICKS) == 29)
ok('lookup by index, full label and suffix agree', PICKS[1] is PICKS['push_edges.cfg246'] is PICKS['cfg246'])

In [ ]:
# 2. Pareto layers as sub-galleries
L = Gallery.open('rd_pareto_layers').split('layer')
print({k: len(g) for k, g in L.items()})
ok('layer 1 is the 3-config front', len(L[1]) == 3)
ok('every PICKS entry with layer k is in layer k', all(e.label in L[e.meta['layer']].labels() for e in PICKS))

In [ ]:
# 3. load: scan-row entries run as the scans did (food_cap=0, mu_egene corrected)
sim, kw = PICKS.load('cfg246', N=64)
ok('food_cap 0, mu_egene 0.01', (sim.food_cap, round(sim.mu_egene, 4)) == (0, 0.01)); sim.free()
G = Gallery.open('rd_regimes')
ok('regime recipes as written (RXD-D4 open): cfg236 food_cap 1', G['cfg236'].effective_params()['food_cap'] == 1)
sim, kw = G.load('cfg236', N=64, food_cap=0); ok('call-time override', sim.food_cap == 0); sim.free()

In [ ]:
# 4. run: parallel, then cached
t = time.time(); R = PICKS[:6].run(N=64, n_steps=300, snaps=(300,), cache=False); t1 = time.time() - t
R1 = PICKS[:6].run(N=64, n_steps=300, snaps=(300,), workers=1, cache=False)
ok('parallel == serial', all(R[i].ts.equals(R1[i].ts) for i in range(6)))
PICKS[:6].run(N=64, n_steps=300, snaps=(300,))
t = time.time(); PICKS[:6].run(N=64, n_steps=300, snaps=(300,)); t2 = time.time() - t
ok(f'cache hit fast ({t1:.2f}s -> {t2:.3f}s)', t2 < 0.5)
R.tiles(cm=1, ncols=6);

In [ ]:
# 5. interactive (opens an SDL window; uncomment to try)
# PICKS.show('cfg246', N=512)
# Gallery.open('rd_waves').show(0, N=512)